# 第 1 周末练习 —— 技术问答解释器（Gemini 流式版）

## 练习目标（理念）

为了展示你对 **Chat Completions API**（这里通过 Gemini 的 OpenAI 兼容端点）以及课程里 **OpenAI / Ollama** 思路的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：用通俗语言讲清楚的解释（面向完全新手）
- **额外要求**：用**流式（streaming）**一边生成一边显示，并做成「打字机」效果

这是你在课程期间自己也能天天用的工具：遇到看不懂的问题，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `gemini.chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放具体问题 |
| 流式输出 `stream=True` | 逐块读 `delta.content`，再用 `update_display` 刷新 |
| 多模型常量 | `MODEL_GPT` / `MODEL_LLAMA` / `MODEL_GEMINI`（本笔记本实际调用 Gemini） |
| OpenAI 兼容端点 | `base_url` 指向 Google Generative Language 的 OpenAI 兼容地址 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `Google_API_KEY`（注意：变量名是本练习自定义的，不是常见的 `OPENAI_API_KEY`）
3. 运行「提问」单元格，在输入框里输入问题，观察流式打字效果


In [3]:
# ========== 导入 + 环境：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 Google API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display / update_display（流式刷新同一块输出）
from IPython.display import Markdown, display, update_display
# 导入标准库 time：打字机效果里用 sleep 控制每个字符刷新间隔
import time

# 加载 .env；override=True 表示用 .env 覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量读取 Google API Key（名字必须和 .env 里一致：Google_API_KEY）
api_key = os.getenv('Google_API_KEY')

# 简单校验：没有 key 就提示；有 key 就打印确认（真正请求在后面单元格）
if not api_key:
    print("No API key was found")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [4]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端小模型名（本笔记本后面主要用 Gemini；这里先定义好常量）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'
# 我正在使用 Gemini-2.5-flash（本练习实际传给 chat.completions.create 的 model）
MODEL_GEMINI = 'gemini-2.5-flash'


In [5]:
# ========== 客户端：用 OpenAI SDK 连 Gemini 的 OpenAI 兼容端点 ==========

# 从 openai 导入 OpenAI 客户端类：即使后端是 Gemini，也可以走同一套 Chat Completions 写法
from openai import OpenAI
# Gemini 的 OpenAI 兼容 base_url（影响请求发往哪里；不要改成别的除非你知道端点变了）
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# 创建客户端：base_url 指向 Google；api_key 用前面读到的 Google_API_KEY
gemini = OpenAI(base_url=base_url,api_key=api_key)


In [6]:
# ========== 用户提示拼装：把「问题」包进发给模型的 user 文本 ==========

# 这是问题；输入此内容以询问新问题
# 函数：接收用户原始 question，拼成发给模型的英文用户提示（影响模型行为的字符串保持英文）
def question_prompt(question):
    # 前缀说明任务 + 拼接用户问题；发给模型的内容保持英文
    question_string = """Please explain this Question: """+ question
    # yield from {book.get("author") for book in books if book.get("author")}
    return question_string
    


In [7]:
# ========== system prompt：定角色与回答风格（发给模型的指令，不翻译） ==========

# system_prompt：告诉模型「你是谁、怎么答」；完整初学者向、通俗解释
system_prompt = """ You are a Technical Assistant who explains the question given to a user 
in an easy language so that someone who is a cmplete beginner can also easily understand it"""


In [8]:
# ========== 主流程：流式问 Gemini，并用 update_display 做打字机效果 ==========

# Get gemini to answer, with 流式输出
def ask_question():
    # 交互输入：在笔记本里弹出输入框，拿到用户问题字符串
    question = input("Enter your Question")
    # 先把问题原样打印出来，方便对照后面的回答
    print(f"Question: {question}")
    # 发起流式 Chat Completions：model / messages / stream=True
    stream = gemini.chat.completions.create(model=MODEL_GEMINI,messages=[
                # system：角色与风格
                {"role": "system", "content": system_prompt},
                # user：真正的问题（经 question_prompt 包装）
                {"role": "user", "content": question_prompt(question)}
            ],
            # stream=True：不要等整段生成完，而是持续返回增量 delta
            stream=True
        )
    # response：累积已生成的完整文本，供 Markdown 刷新显示
    response = ""
    # cursor：打字机光标占位，看起来像还在继续输入
    cursor = "| " 
    # 先占一个空的 Markdown 显示位，拿到 display_id，后面同一块刷新
    display_handle = display(Markdown(""), display_id=True)
    # 遍历流式事件：每个 chunk 可能带一小段 delta.content
    for chunk in stream:
            # 增量文本在 choices[0].delta.content；可能为 None（跳过）
            text = chunk.choices[0].delta.content
            if text:
                # 再按字符刷新：人为放慢，形成打字机观感（不影响 API 返回内容）
                for char in text:
                    response += char
                    # 用同一 display_id 更新 Markdown（正文 + 光标）
                    update_display(
                        Markdown(response + cursor),
                        display_id=display_handle.display_id
                    )
                    # 每个字符停顿约 15ms，控制刷新速度
                    time.sleep(0.015)

    # 全部结束后再刷一次，去掉光标，只保留完整 Markdown
    update_display(
        Markdown(response),
        display_id=display_handle.display_id
    )


In [9]:
# ========== 运行：调用上面的 ask_question，开始交互提问 ==========

# 执行主函数：会弹出输入框，然后流式显示 Gemini 的解释
ask_question()


Question: Can you explain the main characteristics of AI Agents ?


Okay, imagine an "AI Agent" like a **smart worker** or a **smart helper** that lives inside a computer, a robot, or even a software program. Its job is to look at its surroundings (its "environment"), figure things out, and then do something.

The question "Can you explain the main characteristics of AI Agents?" is basically asking:

**"What are the most important things these smart workers (AI Agents) can do or how do they generally behave?"**

Let's break down these "characteristics" (the things they can do) using simple examples:

---

### The Main Characteristics of AI Agents:

1.  **Perception (Sensing)**
    *   **What it means:** An AI Agent needs to "see," "hear," or "feel" what's going on in its world. Just like we use our eyes, ears, and touch to gather information.
    *   **Easy Example:**
        *   A **robot vacuum cleaner (like a Roomba)** "sees" dirt using its sensors.
        *   **Siri or Alexa** "hears" your voice and tries to understand your words.
        *   A **self-driving car** "sees" other cars, traffic lights, and pedestrians using cameras and radar.

2.  **Action (Doing Things)**
    *   **What it means:** After perceiving, the AI Agent needs to *do* something. It doesn't just sit there! It acts upon what it has sensed.
    *   **Easy Example:**
        *   The **robot vacuum** moves around and starts cleaning the dirt it "saw."
        *   **Siri or Alexa** speaks an answer or plays a song after understanding your request.
        *   The **self-driving car** steers, accelerates, or brakes in response to what it "sees" on the road.

3.  **Autonomy (Acting Independently)**
    *   **What it means:** This means the AI Agent can make its *own* decisions and take action without needing a human to tell it what to do every single second. It has some level of independence.
    *   **Easy Example:**
        *   You don't have to control every move of the **robot vacuum**; it figures out its cleaning path by itself.
        *   Once you tell **Siri** a command, it executes it without you having to click buttons for each step.
        *   A **self-driving car** navigates traffic and follows rules without a driver constantly giving instructions.

4.  **Learning (Getting Smarter Over Time)**
    *   **What it means:** Many AI Agents can improve their performance and adapt their behavior based on their past experiences. They get better at their job.
    *   **Easy Example:**
        *   A **robot vacuum** might learn the layout of your house and clean more efficiently over time, or learn where the most dirt usually is.
        *   **Siri or Alexa** might get better at understanding your specific accent or common requests the more you use it.
        *   A **self-driving car** can learn from millions of miles driven, improving its ability to handle difficult situations or new road conditions.

5.  **Goals/Objectives (Having a Purpose)**
    *   **What it means:** An AI Agent doesn't just do random things. It has a specific aim or purpose that it's trying to achieve.
    *   **Easy Example:**
        *   The **robot vacuum's** goal is to clean your floor.
        *   **Siri's** goal is to answer your questions or perform tasks you ask.
        *   The **self-driving car's** goal is to safely get its passengers from point A to point B.

6.  **Rationality (Doing the "Best" Thing)**
    *   **What it means:** An AI Agent tries to make the *best* possible decision to achieve its goals, given the information it has. It tries to be logical and effective.
    *   **Easy Example:**
        *   If the **robot vacuum** sees a very dirty spot, the "rational" action might be to spend more time cleaning that area.
        *   If **Siri** understands you want to know the weather, the "rational" thing is to give you the current weather, not tell a joke.
        *   The **self-driving car** will rationally choose the safest and most efficient lane to drive in, avoiding obstacles.

---

So, in short, the question wants to know how these "smart workers" (AI Agents) operate: **they look at their world, figure things out, make their own decisions, learn from their experiences, and act with a purpose to do the best job possible.**